# 🛣️ Deteksi Kerusakan Jalan — Training YOLO
### Implementasi Model Computer Vision untuk Deteksi Kerusakan Permukaan Jalan pada Infrastruktur Pintar
**SDG 9: Industri, Inovasi & Infrastruktur**

Notebook ini melatih model **YOLO** untuk mendeteksi 3 jenis kerusakan jalan:

| ID | Kelas | Keterangan |
|----|-------|------------|
| 0 | **Pothole** | Lubang jalan |
| 1 | **Crack** | Retakan |
| 2 | **Manhole** | Tutup gorong-gorong |

**Dataset:** [Road Damage Dataset (Kaggle)](https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes) — 2009 gambar, label YOLO siap pakai.

**Output akhir:** file `best.pt` yang bisa langsung dipakai di aplikasi **Streamlit**.

> 💡 Jalankan sel secara berurutan dari atas ke bawah. Untuk training cepat gunakan **GPU** (di Kaggle/Colab: Runtime → pilih GPU).

## 1. Instalasi Dependensi
`ultralytics` = framework YOLO. `kagglehub` = untuk unduh dataset.

In [ ]:
%pip install -q ultralytics kagglehub
print("✅ Instalasi selesai")

In [ ]:
import os, glob, shutil, random, yaml
from pathlib import Path
import ultralytics
from ultralytics import YOLO

ultralytics.checks()   # cek versi + ketersediaan GPU

## 2. Unduh Dataset dari Kaggle
Menggunakan `kagglehub`. Pertama kali dijalankan mungkin diminta login Kaggle (token API).

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes")
print("Path to dataset files:", path)

## 3. Deteksi Struktur Folder Dataset
Struktur di dalam dataset bisa berbeda (kadang ada subfolder `data/`).
Sel ini mencari otomatis folder **images** dan **labels-YOLO** agar notebook tetap jalan.

In [ ]:
def find_dir(root, target_name):
    """Cari folder bernama target_name di dalam root (rekursif)."""
    for p in Path(root).rglob(target_name):
        if p.is_dir():
            return p
    return None

IMAGES_DIR = find_dir(path, "images")
# label YOLO axis-aligned (siap training). fallback ke 'labels' bila tak ada.
LABELS_DIR = find_dir(path, "labels-YOLO") or find_dir(path, "labels")

assert IMAGES_DIR is not None, "Folder images tidak ditemukan!"
assert LABELS_DIR is not None, "Folder labels tidak ditemukan!"

print("📁 Images :", IMAGES_DIR)
print("📁 Labels :", LABELS_DIR)
print("🖼️  Jumlah gambar :", len(list(IMAGES_DIR.glob('*.jpg'))))
print("🏷️  Jumlah label  :", len(list(LABELS_DIR.glob('*.txt'))))

## 4. Eksplorasi Singkat: Distribusi Kelas
Melihat berapa banyak objek per kelas — berguna untuk memahami keseimbangan data.

In [ ]:
from collections import Counter

CLASS_NAMES = {0: "Pothole", 1: "Crack", 2: "Manhole"}
counter = Counter()

for txt in LABELS_DIR.glob("*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            cls = int(line.split()[0])
            counter[cls] += 1

print("Distribusi objek per kelas:")
for cid, name in CLASS_NAMES.items():
    print(f"  {cid} {name:8s}: {counter.get(cid, 0):>5} objek")

## 5. Split Dataset (Train / Val / Test)
YOLO butuh struktur folder:
```
dataset/
 ├── images/{train,val,test}/
 └── labels/{train,val,test}/
```
Kita bagi **80% train, 10% val, 10% test** dengan seed tetap agar hasil reproducible.

In [ ]:
DATASET_DIR = Path("dataset")   # folder kerja di direktori notebook
SPLITS = {"train": 0.8, "val": 0.1, "test": 0.1}
SEED = 42

# ambil hanya gambar yang punya pasangan label
images = sorted([p for p in IMAGES_DIR.glob("*.jpg")
                 if (LABELS_DIR / f"{p.stem}.txt").exists()])
random.seed(SEED)
random.shuffle(images)

n = len(images)
n_train = int(n * SPLITS["train"])
n_val   = int(n * SPLITS["val"])
partition = {
    "train": images[:n_train],
    "val":   images[n_train:n_train + n_val],
    "test":  images[n_train + n_val:],
}

# bersihkan folder lama lalu salin file
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split, files in partition.items():
    img_out = DATASET_DIR / "images" / split
    lbl_out = DATASET_DIR / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img in files:
        shutil.copy(img, img_out / img.name)
        lbl = LABELS_DIR / f"{img.stem}.txt"
        shutil.copy(lbl, lbl_out / lbl.name)
    print(f"  {split:5s}: {len(files)} gambar")

print("✅ Split selesai")

## 6. Buat File Konfigurasi `data.yaml`
File ini memberi tahu YOLO lokasi data dan nama kelas.

In [ ]:
data_config = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASS_NAMES,
}

yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False, allow_unicode=True)

print(yaml_path.read_text())

## 7. Training Model YOLO
Kita pakai **YOLO11n** (nano) — ringan, cepat, cocok untuk demo & deployment.

**Parameter penting:**
- `epochs` — jumlah putaran training (naikkan untuk hasil lebih baik, mis. 100).
- `imgsz` — ukuran input (640 standar).
- `batch` — sesuaikan dengan memori GPU.

> ⏱️ Dengan GPU, 50 epoch ± 20–40 menit. Tanpa GPU akan sangat lambat — kurangi `epochs` untuk uji coba.

In [ ]:
model = YOLO("yolo11n.pt")   # pretrained COCO -> transfer learning

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,          # early stopping bila tak membaik
    project="runs_road",
    name="yolo11n_road",
    plots=True,
)

## 8. Evaluasi Model
Cek metrik pada set validasi: **mAP50**, **mAP50-95**, precision, recall.

In [ ]:
metrics = model.val()
print("mAP50-95 :", round(metrics.box.map, 4))
print("mAP50    :", round(metrics.box.map50, 4))
print("Precision:", round(metrics.box.mp, 4))
print("Recall   :", round(metrics.box.mr, 4))

### Lihat Grafik Hasil Training
Ultralytics menyimpan grafik (loss, mAP, confusion matrix) di folder run.

In [ ]:
from IPython.display import Image, display

run_dir = Path(results.save_dir)
for fname in ["results.png", "confusion_matrix.png"]:
    fpath = run_dir / fname
    if fpath.exists():
        print(f"\n=== {fname} ===")
        display(Image(filename=str(fpath)))

## 9. Uji Inferensi pada Gambar Test
Coba prediksi beberapa gambar dan tampilkan hasil deteksinya.

In [ ]:
import matplotlib.pyplot as plt

test_images = list((DATASET_DIR / "images" / "test").glob("*.jpg"))[:3]

for img_path in test_images:
    res = model.predict(str(img_path), conf=0.25, verbose=False)[0]
    print(f"🖼️ {img_path.name} — {len(res.boxes)} deteksi")
    for box in res.boxes:
        print(f"   {CLASS_NAMES[int(box.cls)]:8s}  conf={float(box.conf):.2f}")

    annotated = res.plot()[..., ::-1]   # BGR -> RGB
    plt.figure(figsize=(9, 5))
    plt.imshow(annotated)
    plt.axis("off")
    plt.title(img_path.name)
    plt.show()

## 10. Export Model untuk Deployment 🚀
Salin `best.pt` ke folder `models/` supaya bisa langsung dipakai aplikasi Streamlit.

Opsional: export ke **ONNX** untuk inferensi lebih ringan tanpa PyTorch.

In [ ]:
best_weight = run_dir / "weights" / "best.pt"
print("Bobot terbaik:", best_weight)

# salin ke folder models/ (dipakai oleh app Streamlit)
Path("models").mkdir(exist_ok=True)
shutil.copy(best_weight, "models/best.pt")
print("✅ Tersalin ke models/best.pt")

# (opsional) export ONNX
# model.export(format="onnx")

## ✅ Selesai — Langkah Berikutnya

1. **Unduh** file `models/best.pt` dari notebook ini.
2. Buka folder aplikasi Streamlit (`road-damage-app/`).
3. Letakkan `best.pt` ke dalam `road-damage-app/models/`.
4. Jalankan lokal:
   ```bash
   cd road-damage-app
   pip install -r requirements.txt
   streamlit run app.py
   ```
5. Deploy gratis ke **[Streamlit Community Cloud](https://share.streamlit.io)** — hubungkan repo GitHub, arahkan ke `app.py`.

> 📌 **Catatan:** untuk hasil deteksi lebih akurat, naikkan `epochs` (mis. 100–150) dan/atau gunakan model lebih besar (`yolo11s.pt`, `yolo11m.pt`).